# Licenciatura en Ciencia de Datos
## Calidad y Preprocesamiento de Datos

---

### **Proyecto Final — Análisis y Visualización**

**Equipo**

**Castrillo Cruz Karen Arlet**

**Pérez Aguiar Oropeza Gabriel Emiliano**

**Ramos González Nadia**

**Rueda Reyes Fabián**

**Torres Pasión Angel Isaac**


### **Objetivo**

Generar 10 figuras compuestas a partir de los datos procesados en los notebooks anteriores (`perfilado`, `limpieza`, `fusion`). Cada figura responde a preguntas analíticas específicas sobre violencia de género en México (2020–2022), cruzando las fuentes INE y BANAVIM para hacer visible lo que los datos oficiales ocultan.

**Fuentes procesadas:**
- `../Data/ine_incidencias_2020_2022.csv` — 162 registros limpios
- `../Data/banavim_fix_2020_2022.csv` — 806 092 registros sin mojibake
- `../Data/fusion_outputs/perfil_estado_integrado.csv`
- `../Data/fusion_outputs/matches_snm.csv`


## 0. Configuración del entorno


In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

print('Entorno listo')


Entorno listo


### 0.1 Paleta de colores y estilo global


In [2]:
# Paleta del proyecto
CRIMSON   = '#8B1A1A'   # INE / violencia política
CRIMSON_L = '#C0392B'
TEAL      = '#0E4D64'   # BANAVIM / violencia doméstica
TEAL_L    = '#1A7A9E'
GOLD      = '#7D6608'   # riesgo integrado
GOLD_L    = '#D4AC0D'
SLATE     = '#2C3E50'   # texto neutral
GRAY_BG   = '#F7F7F5'
GRAY_GRID = '#E0E0E0'

# Colormaps personalizados
CMAP_RISK = mcolors.LinearSegmentedColormap.from_list(
    'risk', ['#FEF9E7', '#F9E79F', '#F39C12', '#E74C3C', '#7B241C'])
CMAP_BLUE = mcolors.LinearSegmentedColormap.from_list(
    'blue_seq', ['#EAF2FF', '#85B7EB', '#185FA5', '#042C53'])

def estilo_base():
    plt.rcParams.update({
        'figure.facecolor':    'white',
        'axes.facecolor':      GRAY_BG,
        'axes.grid':           True,
        'grid.alpha':          0.4,
        'grid.color':          GRAY_GRID,
        'grid.linestyle':      '--',
        'grid.linewidth':      0.5,
        'axes.spines.top':     False,
        'axes.spines.right':   False,
        'axes.spines.left':    False,
        'axes.spines.bottom':  True,
        'axes.linewidth':      0.5,
        'xtick.bottom':        True,
        'ytick.left':          False,
        'xtick.color':         SLATE,
        'ytick.color':         SLATE,
        'font.family':         'DejaVu Sans',
        'text.color':          SLATE,
        'axes.labelcolor':     SLATE,
        'figure.dpi':          150,
        'axes.titlesize':      11,
        'axes.titleweight':    'bold',
        'axes.titlepad':       10,
        'axes.labelsize':      9,
        'xtick.labelsize':     8,
        'ytick.labelsize':     8,
        'legend.fontsize':     8,
        'legend.framealpha':   0.8,
        'savefig.bbox':        'tight',
        'savefig.facecolor':   'white',
    })

estilo_base()
print('Estilo aplicado')


Estilo aplicado


### 0.2 Funciones utilitarias


In [3]:
# Directorio de salida para figuras
OUT = Path('../Data/graficas')
OUT.mkdir(parents=True, exist_ok=True)

def guardar(nombre):
    p = OUT / f'{nombre}.png'
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close('all')
    print(f'  OK  {p.name}')

def titulo_figura(fig, titulo, subtitulo=''):
    fig.suptitle(titulo, fontsize=14, fontweight='bold', color=SLATE,
                 y=1.01, x=0.02, ha='left')
    if subtitulo:
        fig.text(0.02, 0.995, subtitulo, fontsize=8.5, color='#7F8C8D',
                 style='italic', ha='left', va='top')

def anotar(ax, texto, x, y, color=SLATE, fs=7.5, ha='left', bold=False):
    ax.annotate(texto, xy=(x, y), xycoords='data',
                fontsize=fs, color=color, ha=ha, va='center',
                fontweight='bold' if bold else 'normal')

def lollipop(ax, values, labels, color, size=8, annot=True):
    y = np.arange(len(values))
    ax.hlines(y, 0, values, colors=color, alpha=0.35, lw=2)
    ax.plot(values, y, 'o', color=color, ms=size, zorder=5,
            markeredgecolor='white', markeredgewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    if annot:
        for v, yi in zip(values, y):
            ax.annotate(f'{v:,.0f}', xy=(v, yi), xytext=(4, 0),
                        textcoords='offset points', va='center',
                        fontsize=7.5, color=color)

def safe_str(s):
    return s.fillna('').astype(str)

print('Funciones utilitarias definidas')


Funciones utilitarias definidas


## 1. Carga de datos


In [4]:
print('Cargando datos...')
ine    = pd.read_csv('../Data/ine_incidencias_2020_2022.csv', low_memory=False)
bv     = pd.read_csv('../Data/banavim_fix_2020_2022.csv',    low_memory=False)
perfil = pd.read_csv('../Data/fusion_outputs/perfil_estado_integrado.csv')

print(f'  INE: {ine.shape} | BANAVIM: {bv.shape} | perfil: {perfil.shape}')


Cargando datos...
  INE: (162, 25) | BANAVIM: (806092, 36) | perfil: (12, 8)


### 1.1 Columnas clave y golden record INE


In [5]:
# Columnas key — compatibilidad con distintas versiones del CSV
INE_ENT   = 'Entidad Federativa'
INE_SANC  = 'Sancion' if 'Sancion' in ine.columns else 'Sanción'
INE_REL   = 'Relacion Con La Victima' if 'Relacion Con La Victima' in ine.columns else 'Relación Con La Víctima'
INE_SEX   = 'Sexo'
INE_TIPO  = 'tipo_violencia'
INE_REIN  = 'Reincidencia De La Conducta'
INE_FECHA = 'fecha_resolucion'
BV_EDAD   = 'edad'
BV_ARMA   = 'Portada Dicha Arma' if 'Portada Dicha Arma' in bv.columns else 'Portaba Dicha Arma'
BV_ALCO   = 'Droga_Alcohol'
BV_ESCOL  = 'escolaridad'
BV_VINC   = 'vinculo_victima'
BV_ESTADO = 'estado'

# Golden record: consolidar 162 incidencias en individuos únicos
golden = (ine.groupby('nombre', dropna=True)
          .agg(
              entidad       = (INE_ENT,   'first'),
              sexo          = (INE_SEX,   'first'),
              n_incidencias = ('nombre',  'count'),
              sanciones     = (INE_SANC,  lambda x: ' | '.join(x.dropna().astype(str).unique())),
          )
          .reset_index()
         )
golden['es_reincidente'] = golden['n_incidencias'] > 1

print(f'  Golden record: {len(golden)} individuos únicos')
print(f'  Reincidentes:  {golden["es_reincidente"].sum()} ({golden["es_reincidente"].mean()*100:.0f}%)')


  Golden record: 140 individuos únicos
  Reincidentes:  12 (9%)


## 2. Figuras — Análisis INE

Las primeras figuras exploran los datos del INE: quiénes son sancionados, qué tipos de violencia se registran y cómo se distribuyen territorialmente.


### Fig. 1 — Perfil de personas sancionadas (INE)


In [6]:
print('Fig 1 — Perfil de sancionados (INE)...')

fig = plt.figure(figsize=(18, 6))
gs  = GridSpec(1, 3, figure=fig, wspace=0.38)
titulo_figura(fig,
    'Perfil de personas sancionadas por violencia política de género — INE (2020–2022)',
    'Golden record: 162 incidencias consolidadas en 140 individuos únicos · '
    'Limpieza: aliases removidos, nombres normalizados, reincidentes identificados')

# Panel A: top estados
ax1 = fig.add_subplot(gs[0])
top_estados = ine[INE_ENT].value_counts().head(12).sort_values()
lollipop(ax1, top_estados.values, top_estados.index, CRIMSON)
ax1.set_title('Sanciones por estado', fontweight='bold', color=CRIMSON)
ax1.set_xlabel('Número de sanciones')
ax1.axvline(top_estados.mean(), color=GOLD_L, lw=1.2, ls='--', alpha=0.7,
            label=f'Media: {top_estados.mean():.1f}')
ax1.legend()
ax1.text(0.98, 0.02, f'Oaxaca lidera con\n{top_estados.max()} sanciones',
         transform=ax1.transAxes, ha='right', va='bottom',
         fontsize=7.5, color=CRIMSON, style='italic')

# Panel B: tipo de sanción
ax2 = fig.add_subplot(gs[1])
sanciones   = safe_str(ine[INE_SANC]).value_counts()
sanciones   = sanciones[sanciones.index != ''].head(8).sort_values()
colores_s   = [CRIMSON if 'Ninguna' in s else TEAL for s in sanciones.index]
bars = ax2.barh(range(len(sanciones)), sanciones.values,
                color=colores_s, edgecolor='white', height=0.7)
ax2.set_yticks(range(len(sanciones)))
ax2.set_yticklabels([s[:30] for s in sanciones.index], fontsize=8)
ax2.set_title('Tipo de sanción impuesta', fontweight='bold', color=TEAL)
ax2.set_xlabel('Frecuencia')
for bar, val in zip(bars, sanciones.values):
    ax2.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=7.5, color=SLATE)
pct_ninguna = (sanciones.get('Ninguna', 0) / sanciones.sum() * 100)
ax2.text(0.98, 0.02,
         f'{pct_ninguna:.0f}% recibe\nsanción "Ninguna"',
         transform=ax2.transAxes, ha='right', va='bottom',
         fontsize=7.5, color=CRIMSON, style='italic', fontweight='bold')

# Panel C: sexo y reincidencia
ax3 = fig.add_subplot(gs[2])
sexo   = safe_str(ine[INE_SEX]).replace('', 'No especificado').value_counts()
sizes  = sexo.values
labels = [f'{l}\n({v})' for l, v in zip(sexo.index, sexo.values)]
colors = [CRIMSON, TEAL, GOLD][:len(sizes)]
wedges, texts, autotexts = ax3.pie(
    sizes, labels=labels, autopct='%1.0f%%',
    colors=colors, startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
    textprops=dict(fontsize=8))
for at in autotexts:
    at.set_fontsize(8.5)
    at.set_fontweight('bold')
    at.set_color('white')
ax3.set_title('Sexo del agresor sancionado\n(INE)', fontweight='bold', color=SLATE)
n_rein = golden['es_reincidente'].sum()
ax3.text(0, -1.35,
         f'{n_rein} individuos reincidentes · '
         f'{n_rein/len(golden)*100:.0f}% del total',
         ha='center', fontsize=8, color=CRIMSON, fontweight='bold')

plt.tight_layout()
guardar('fig01_perfil_sancionados_ine')


Fig 1 — Perfil de sancionados (INE)...
  OK  fig01_perfil_sancionados_ine.png


### Fig. 2 — Tipos de violencia y relación con la víctima (INE)


In [7]:
print('Fig 2 — Tipos de violencia...')

fig = plt.figure(figsize=(18, 7))
gs  = GridSpec(1, 2, figure=fig, wspace=0.4)
titulo_figura(fig,
    '¿Qué tipo de violencia registra el INE y contra quién?',
    'Limpieza: campo tipo_violencia extraído de XML crudo (<element>...</element>) · '
    'Relación con la víctima normalizada')

# Panel A: tipos de violencia (después de extraer XML)
ax1 = fig.add_subplot(gs[0])
tipos = (safe_str(ine[INE_TIPO])
         .str.split(' | ', regex=False)
         .explode()
         .str.strip()
         .replace('', np.nan)
         .dropna()
         .value_counts()
         .head(10))
lollipop(ax1, tipos.sort_values().values,
         [t[:32] for t in tipos.sort_values().index], CRIMSON)
ax1.set_title('Tipos de violencia\n(extraídos de XML · INE 2020–2022)',
              fontweight='bold', color=CRIMSON)
ax1.set_xlabel('Frecuencia')
ax1.text(0.98, 0.02,
         'La violencia simbólica\nes la más registrada',
         transform=ax1.transAxes, ha='right', va='bottom',
         fontsize=7.5, color=CRIMSON, style='italic')

# Panel B: heatmap violencia × sanción
ax2 = fig.add_subplot(gs[1])
tipos_exp = ine[[INE_TIPO, INE_SANC]].copy()
tipos_exp[INE_TIPO] = safe_str(tipos_exp[INE_TIPO]).str.split(' | ', regex=False)
tipos_exp = tipos_exp.explode(INE_TIPO)
tipos_exp[INE_TIPO] = tipos_exp[INE_TIPO].str.strip().replace('', np.nan)
tipos_exp = tipos_exp.dropna()

top_t = tipos_exp[INE_TIPO].value_counts().head(7).index
top_s = tipos_exp[INE_SANC].value_counts().head(5).index
tipos_filtrado = tipos_exp[
    tipos_exp[INE_TIPO].isin(top_t) &
    tipos_exp[INE_SANC].isin(top_s)
].copy()
ct = pd.crosstab(
    tipos_filtrado[INE_TIPO],
    tipos_filtrado[INE_SANC]
).reindex(top_t).fillna(0)
mask = ct == 0
sns.heatmap(ct, annot=True, fmt='.0f', cmap='Reds',
            linewidths=0.5, ax=ax2, mask=mask,
            cbar_kws={'label': 'Número de casos', 'shrink': 0.8},
            annot_kws={'size': 8})
sns.heatmap(ct, annot=False, fmt='', cmap=['#F5F5F5'],
            linewidths=0.5, ax=ax2, mask=~mask,
            cbar=False)
ax2.set_title('Tipo de violencia × Sanción impuesta\n(INE 2020–2022)',
              fontweight='bold', color=TEAL)
ax2.set_xlabel('Sanción', fontsize=9)
ax2.set_ylabel('Tipo de violencia', fontsize=9)
ax2.tick_params(axis='x', rotation=30, labelsize=7.5)
ax2.tick_params(axis='y', rotation=0,  labelsize=7.5)

plt.tight_layout()
guardar('fig02_tipos_violencia_sanciones')


Fig 2 — Tipos de violencia...
  OK  fig02_tipos_violencia_sanciones.png


### Fig. 3 — Concentración territorial comparada


In [8]:
print('Fig 3 — Territorio...')

fig = plt.figure(figsize=(18, 8))
gs  = GridSpec(1, 3, figure=fig, wspace=0.4)
titulo_figura(fig,
    '¿Dónde ocurre la violencia de género? Distribución territorial comparada',
    'INE: sanciones formales por entidad · '
    'BANAVIM: casos registrados por estado de residencia (escala logarítmica)')

# Panel A: INE por estado (lollipop)
ax1 = fig.add_subplot(gs[0])
ine_estado = ine[INE_ENT].value_counts().head(15).sort_values()
lollipop(ax1, ine_estado.values, ine_estado.index, CRIMSON)
ax1.set_title('Sanciones INE\npor entidad federativa (top 15)',
              fontweight='bold', color=CRIMSON)
ax1.set_xlabel('Número de sanciones')

# Panel B: BANAVIM por estado — escala logarítmica
ax2 = fig.add_subplot(gs[1])
bv_estado    = safe_str(bv[BV_ESTADO]).replace('', np.nan).dropna()
bv_estado_ct = bv_estado.value_counts().head(15).sort_values()
bars = ax2.barh(range(len(bv_estado_ct)), bv_estado_ct.values,
                color=TEAL, edgecolor='white', height=0.7, alpha=0.85)
ax2.set_yticks(range(len(bv_estado_ct)))
ax2.set_yticklabels(bv_estado_ct.index, fontsize=8)
ax2.set_xscale('log')
ax2.set_title('Casos BANAVIM por estado\n(escala logarítmica · top 15)',
              fontweight='bold', color=TEAL)
ax2.set_xlabel('Número de casos (log)')
ax2.xaxis.set_major_formatter(
    mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(bars, bv_estado_ct.values):
    ax2.text(val * 1.05, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=6.5, color=TEAL)

# Panel C: bubble chart INE vs BANAVIM
ax3 = fig.add_subplot(gs[2])
comp_terr = (perfil.groupby('estado_ine')
             .agg(sanciones=('n_sanciones_ine','sum'),
                  casos=('n_casos_banavim','sum'),
                  riesgo=('pct_portaba_arma','mean'))
             .reset_index()
             .dropna())
sc = ax3.scatter(comp_terr['sanciones'] + 0.1,
                 np.log10(comp_terr['casos'] + 1),
                 c=comp_terr['riesgo'],
                 cmap=CMAP_RISK,
                 s=comp_terr['casos'] / comp_terr['casos'].max() * 400 + 30,
                 alpha=0.85, edgecolors=SLATE, linewidths=0.5)
for _, row in comp_terr.iterrows():
    ax3.annotate(row['estado_ine'][:8],
                 (row['sanciones'] + 0.1, np.log10(row['casos'] + 1)),
                 xytext=(3, 3), textcoords='offset points',
                 fontsize=6.5, color=SLATE)
plt.colorbar(sc, ax=ax3, label='% agresores con arma (BANAVIM)', shrink=0.8)
ax3.set_title('Sanciones INE vs casos BANAVIM\n(tamaño = volumen BANAVIM)',
              fontweight='bold', color=SLATE)
ax3.set_xlabel('Sanciones INE')
ax3.set_ylabel('Casos BANAVIM (log₁₀)')
ax3.text(0.02, 0.97,
         'Quintana Roo:\n1 sanción · 1 255 casos',
         transform=ax3.transAxes, va='top', fontsize=7, color=CRIMSON,
         style='italic',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
guardar('fig03_distribucion_territorial')


Fig 3 — Territorio...
  OK  fig03_distribucion_territorial.png


### Fig. 4 — Vínculos víctima-agresor comparados


In [9]:
print('Fig 4 — Vínculos...')

fig = plt.figure(figsize=(18, 7))
gs  = GridSpec(1, 3, figure=fig, wspace=0.42)
titulo_figura(fig,
    '¿Quién agrede a quién? Relaciones víctima-agresor en ambas fuentes',
    'INE: categorías de poder político (LGAMVLV Art. 20 Bis) · '
    'BANAVIM: categorías de relación personal (NOM-046) · '
    'Homologación semántica realizada para comparar')

# Panel A: INE vínculos
ax1 = fig.add_subplot(gs[0])
rel_ine = (safe_str(ine[INE_REL]).replace('', np.nan)
           .dropna().value_counts().head(8).sort_values())
lollipop(ax1, rel_ine.values,
         [r[:28] for r in rel_ine.index], CRIMSON)
ax1.set_title('Vínculo agresor-víctima\nINE — violencia política',
              fontweight='bold', color=CRIMSON)
ax1.set_xlabel('Frecuencia')

# Panel B: BANAVIM vínculos
ax2 = fig.add_subplot(gs[1])
rel_bv = (safe_str(bv[BV_VINC]).str.upper().replace('', np.nan)
          .dropna().value_counts().head(8).sort_values())
lollipop(ax2, rel_bv.values,
         [r[:28] for r in rel_bv.index], TEAL)
ax2.set_title('Vínculo agresor-víctima\nBANAVIM — violencia doméstica',
              fontweight='bold', color=TEAL)
ax2.set_xlabel('Frecuencia (escala log)')
ax2.set_xscale('log')
ax2.xaxis.set_major_formatter(
    mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Panel C: grupos homologados comparados
ax3 = fig.add_subplot(gs[2])
vinculos_homol = (perfil.groupby('vinculo_grupo_ine')
                  .agg(ine_sanc=('n_sanciones_ine','sum'),
                       bv_casos=('n_casos_banavim','sum'))
                  .reset_index()
                  .sort_values('bv_casos', ascending=True))
y = np.arange(len(vinculos_homol))
ax3.barh(y - 0.2, vinculos_homol['ine_sanc'], 0.35,
         color=CRIMSON, label='INE (sanciones)', edgecolor='white', alpha=0.9)
ax3_b = ax3.twiny()
ax3_b.barh(y + 0.2, vinculos_homol['bv_casos'], 0.35,
           color=TEAL, label='BANAVIM (casos)', edgecolor='white', alpha=0.7)
ax3.set_yticks(y)
ax3.set_yticklabels(vinculos_homol['vinculo_grupo_ine'], fontsize=8.5)
ax3.set_xlabel('Sanciones INE', color=CRIMSON, fontsize=9)
ax3_b.set_xlabel('Casos BANAVIM', color=TEAL, fontsize=9)
ax3.set_title('Grupos homologados\nINE vs BANAVIM',
              fontweight='bold', color=SLATE)
handles = [mpatches.Patch(color=CRIMSON, label='INE'),
           mpatches.Patch(color=TEAL,    label='BANAVIM')]
ax3.legend(handles=handles, loc='lower right', fontsize=8)
ax3.text(0.02, -0.14,
         'Los vocabularios son distintos: INE usa poder político,\n'
         'BANAVIM usa relación personal. Comparación semántica aproximada.',
         transform=ax3.transAxes, fontsize=7, color='#7F8C8D', style='italic')

plt.tight_layout()
guardar('fig04_vinculos_victima_agresor')


Fig 4 — Vínculos...
  OK  fig04_vinculos_victima_agresor.png


## 3. Figuras — Análisis BANAVIM

Las siguientes figuras profundizan en el perfil sociodemográfico del agresor a partir de los 806 092 registros del BANAVIM.


### Fig. 5 — Perfil sociodemográfico del agresor (BANAVIM)


In [10]:
print('Fig 5 — Perfil agresor BANAVIM...')

fig = plt.figure(figsize=(18, 8))
gs  = GridSpec(2, 3, figure=fig, wspace=0.4, hspace=0.5)
titulo_figura(fig,
    'Perfil sociodemográfico del agresor — BANAVIM (806 092 registros)',
    'Datos procesados: mojibake reparado con ftfy · '
    'edades atípicas (<14 o >85 años) excluidas · valores semánticos nulos eliminados')

bv_limpio = bv.copy()
bv_limpio['portaba_arma'] = (safe_str(bv_limpio[BV_ARMA]).str.upper() == 'SI').astype(int)
bv_limpio['alcohol']      = (safe_str(bv_limpio[BV_ALCO]).str.upper() == 'SI').astype(int)

# P1: distribución de edad (KDE + histograma)
ax1 = fig.add_subplot(gs[0, :2])
edad_clean = pd.to_numeric(bv_limpio[BV_EDAD], errors='coerce').dropna()
edad_clean = edad_clean[edad_clean.between(14, 85)]
n, bins, patches = ax1.hist(edad_clean, bins=40,
                             color=TEAL, edgecolor='white',
                             alpha=0.6, density=True)
edad_clean.plot.kde(ax=ax1, color=TEAL_L, lw=2.5, label='Densidad KDE')
ax1.axvline(edad_clean.mean(), color=CRIMSON, lw=1.8, ls='--',
            label=f'Media: {edad_clean.mean():.1f} años')
ax1.axvline(edad_clean.median(), color=GOLD_L, lw=1.8, ls=':',
            label=f'Mediana: {edad_clean.median():.1f} años')
ax1.set_title('Distribución de edad del agresor (n≈661 k con edad válida)',
              fontweight='bold', color=TEAL)
ax1.set_xlabel('Edad')
ax1.set_ylabel('Densidad')
ax1.legend()
p25, p75 = edad_clean.quantile([0.25, 0.75])
ax1.axvspan(p25, p75, alpha=0.08, color=TEAL_L,
            label=f'IQR: {p25:.0f}–{p75:.0f} años')
ax1.text(0.97, 0.95,
         f'50% de los agresores\ntiene entre {p25:.0f} y {p75:.0f} años',
         transform=ax1.transAxes, ha='right', va='top',
         fontsize=8, color=TEAL,
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.9))

# P2: escolaridad
ax2 = fig.add_subplot(gs[0, 2])
escol    = safe_str(bv_limpio[BV_ESCOL]).replace('', np.nan).dropna()
escol_ct = escol.str.upper().value_counts().head(7).sort_values()
ax2.barh(range(len(escol_ct)), escol_ct.values,
         color=[TEAL if i > len(escol_ct)//2 else '#7FB3D3'
                for i in range(len(escol_ct))],
         edgecolor='white', height=0.7)
ax2.set_yticks(range(len(escol_ct)))
ax2.set_yticklabels([e[:22] for e in escol_ct.index], fontsize=8)
ax2.set_title('Escolaridad del agresor', fontweight='bold', color=TEAL)
ax2.set_xlabel('Frecuencia')
for i, val in enumerate(escol_ct.values):
    ax2.text(val + escol_ct.max()*0.01, i, f'{val/escol_ct.sum()*100:.0f}%',
             va='center', fontsize=7, color=TEAL)

# P3: portación de arma por vínculo
ax3 = fig.add_subplot(gs[1, 0])
arma_vinc = (bv_limpio.assign(vinculo=safe_str(bv_limpio[BV_VINC]).str.upper())
             .groupby('vinculo')['portaba_arma']
             .agg(['mean','count'])
             .query('count >= 500')
             .sort_values('mean', ascending=False)
             .head(8)
             .reset_index())
arma_vinc['pct'] = arma_vinc['mean'] * 100
bars = ax3.barh(range(len(arma_vinc)), arma_vinc['pct'],
                color=[CRIMSON if p > 10 else '#D98880'
                       for p in arma_vinc['pct']],
                edgecolor='white', height=0.7)
ax3.set_yticks(range(len(arma_vinc)))
ax3.set_yticklabels([v[:22] for v in arma_vinc['vinculo']], fontsize=7.5)
ax3.xaxis.set_major_formatter(mtick.PercentFormatter())
ax3.set_title('% con arma por vínculo\n(grupos ≥500 casos)',
              fontweight='bold', color=CRIMSON)
for bar, pct in zip(bars, arma_vinc['pct']):
    ax3.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
             f'{pct:.1f}%', va='center', fontsize=7, color=CRIMSON)

# P4: consumo alcohol+drogas por escolaridad
ax4 = fig.add_subplot(gs[1, 1])
drogas_escol = (bv_limpio
    .assign(escol=safe_str(bv_limpio[BV_ESCOL]).str.upper().replace('', np.nan))
    .dropna(subset=['escol'])
    .groupby('escol')
    .agg(pct_alco=('alcohol','mean'),
         n=('alcohol','count'))
    .query('n >= 200')
    .sort_values('pct_alco', ascending=True)
    .head(8)
    .reset_index())
drogas_escol['pct_alco_pct'] = drogas_escol['pct_alco'] * 100
lollipop(ax4, drogas_escol['pct_alco_pct'],
         [e[:22] for e in drogas_escol['escol']], GOLD, size=7)
ax4.xaxis.set_major_formatter(mtick.PercentFormatter())
ax4.set_title('% consumo de alcohol\npor nivel educativo',
              fontweight='bold', color=GOLD)
ax4.set_xlabel('% con alcohol en la agresión')

# P5: violín edad por portación de arma
ax5 = fig.add_subplot(gs[1, 2])
bv_violin = bv_limpio.copy()
bv_violin['edad_num'] = pd.to_numeric(bv_violin[BV_EDAD], errors='coerce')
bv_violin = bv_violin.dropna(subset=['edad_num'])
bv_violin = bv_violin[bv_violin['edad_num'].between(14, 85)]
bv_violin['grupo_arma'] = bv_violin['portaba_arma'].map(
    {1: 'Con arma', 0: 'Sin arma'})
grupos = [bv_violin[bv_violin['grupo_arma'] == g]['edad_num'].values
          for g in ['Con arma', 'Sin arma']]
vp = ax5.violinplot(grupos, positions=[1, 2],
                    showmedians=True, showextrema=False)
for i, (body, color) in enumerate(zip(vp['bodies'], [CRIMSON, TEAL])):
    body.set_facecolor(color)
    body.set_alpha(0.7)
vp['cmedians'].set_color([CRIMSON, TEAL])
vp['cmedians'].set_linewidth(2.5)
ax5.set_xticks([1, 2])
ax5.set_xticklabels(['Con arma', 'Sin arma'], fontsize=9)
ax5.set_ylabel('Edad del agresor')
ax5.set_title('Distribución de edad\npor portación de arma',
              fontweight='bold', color=SLATE)
med_arma  = np.median(grupos[0])
med_noarm = np.median(grupos[1])
ax5.text(0.5, 0.05,
         f'Mediana con arma: {med_arma:.0f} años\n'
         f'Mediana sin arma: {med_noarm:.0f} años',
         transform=ax5.transAxes, ha='center', fontsize=7.5, color=SLATE,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

plt.tight_layout()
guardar('fig05_perfil_agresor_banavim')


Fig 5 — Perfil agresor BANAVIM...
  OK  fig05_perfil_agresor_banavim.png


## 4. Figuras — Índice de riesgo y consultas procesadas

Las figuras 6–8 construyen el índice de riesgo territorial y responden a las consultas analíticas diseñadas para cada fuente.


### Fig. 6 — Índice de riesgo territorial integrado


In [11]:
print('Fig 6 — Índice de riesgo...')

# Construir índice ponderado con MinMaxScaler
perfil_idx  = perfil.copy()
vars_riesgo = ['n_sanciones_ine', 'n_casos_banavim', 'pct_portaba_arma', 'pct_alcohol']
scaler = MinMaxScaler()
tmp = scaler.fit_transform(perfil_idx[vars_riesgo].fillna(0))
perfil_idx['indice'] = (tmp[:,0]*0.30 + tmp[:,1]*0.40 +
                        tmp[:,2]*0.20 + tmp[:,3]*0.10).round(3)

ranking = (perfil_idx.groupby('estado_ine')['indice']
           .mean().sort_values(ascending=False).reset_index())
ranking.columns = ['estado', 'indice']

fig = plt.figure(figsize=(18, 8))
gs  = GridSpec(1, 2, figure=fig, wspace=0.4)
titulo_figura(fig,
    'Índice de riesgo territorial integrado — INE + BANAVIM',
    'Ponderación: casos BANAVIM 40% · sanciones INE 30% · '
    '% con arma 20% · % con alcohol 10% · normalización MinMaxScaler')

# Panel A: ranking por estado
ax1 = fig.add_subplot(gs[0])
n     = len(ranking)
norm  = mcolors.Normalize(vmin=0, vmax=ranking['indice'].max())
colores_r = [CMAP_RISK(norm(v)) for v in ranking['indice']]
bars = ax1.barh(range(n), ranking['indice'],
                color=colores_r, edgecolor='white', height=0.75)
ax1.set_yticks(range(n))
ax1.set_yticklabels(ranking['estado'], fontsize=9)
ax1.set_xlabel('Índice de riesgo compuesto (0–1)')
ax1.set_title('Ranking de riesgo territorial\npor estado',
              fontweight='bold', color=SLATE)
for bar, val in zip(bars, ranking['indice']):
    ax1.text(val + 0.003, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=7.5)
media = ranking['indice'].mean()
ax1.axvline(media, color=GOLD, lw=1.5, ls='--', alpha=0.7,
            label=f'Media: {media:.3f}')
ax1.legend(fontsize=8)
sm = plt.cm.ScalarMappable(cmap=CMAP_RISK, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax1, label='Nivel de riesgo', shrink=0.5,
             orientation='horizontal', pad=0.08)

# Panel B: heatmap estado × vínculo
ax2 = fig.add_subplot(gs[1])
pivot = (perfil_idx
         .pivot_table(index='estado_ine', columns='vinculo_grupo_ine',
                      values='indice', aggfunc='mean')
         .fillna(0))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap=CMAP_RISK,
            linewidths=0.5, ax=ax2,
            annot_kws={'size': 8},
            cbar_kws={'label': 'Índice de riesgo', 'shrink': 0.8})
ax2.set_title('Índice por estado y tipo de vínculo\n(cruza ambas fuentes)',
              fontweight='bold', color=SLATE)
ax2.set_xlabel('Tipo de vínculo (grupos homologados)', fontsize=9)
ax2.set_ylabel('Estado', fontsize=9)
ax2.tick_params(axis='x', rotation=30, labelsize=7.5)
ax2.tick_params(axis='y', rotation=0,  labelsize=8)
ax2.text(0.02, -0.18,
         'Quintana Roo (OTRO) y Tabasco (OTRO) destacan por alto volumen BANAVIM '
         'con poca sanción formal — señal de subregistro institucional.',
         transform=ax2.transAxes, fontsize=7.5, color=CRIMSON, style='italic')

plt.tight_layout()
guardar('fig06_indice_riesgo_territorial')


Fig 6 — Índice de riesgo...
  OK  fig06_indice_riesgo_territorial.png


### Fig. 7 — 5 Consultas INE (datos procesados)


In [12]:
print('Fig 7 — 5 consultas INE...')

fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(2, 3, figure=fig, wspace=0.42, hspace=0.55)
titulo_figura(fig,
    '5 Consultas sobre el INE — datos después del procesamiento',
    'Golden record · alias removidos · fechas parseadas · XML extraído · '
    'columnas con <20% completitud descartadas')

# C_INE_1: Evolución temporal de resoluciones
ax1 = fig.add_subplot(gs[0, 0])
ine['anio']  = pd.to_datetime(safe_str(ine[INE_FECHA]), errors='coerce').dt.year
ine_anual    = ine['anio'].value_counts().sort_index()
ax1.fill_between(ine_anual.index, ine_anual.values, color=CRIMSON, alpha=0.35)
ax1.plot(ine_anual.index, ine_anual.values,
         color=CRIMSON, lw=2.5, marker='o', ms=8,
         markeredgecolor='white', markeredgewidth=1.5)
for x, y in zip(ine_anual.index, ine_anual.values):
    ax1.annotate(f'{y}', (x, y), xytext=(0, 8),
                 textcoords='offset points', ha='center',
                 fontsize=9, color=CRIMSON, fontweight='bold')
ax1.set_title('C_INE_1 · Resoluciones por año\n(INE 2020–2022)',
              fontweight='bold', color=CRIMSON)
ax1.set_xlabel('Año')
ax1.set_ylabel('Resoluciones')
ax1.set_xticks(ine_anual.index)

# C_INE_2: Reincidencia en el golden record
ax2 = fig.add_subplot(gs[0, 1])
n_rein = golden['es_reincidente'].sum()
n_uni  = (~golden['es_reincidente']).sum()
wedges, texts, auto = ax2.pie(
    [n_rein, n_uni],
    labels=[f'Reincidentes\n({n_rein})', f'Una sola vez\n({n_uni})'],
    autopct='%1.0f%%',
    colors=[CRIMSON, '#BDC3C7'],
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=9))
for at in auto:
    at.set_fontsize(9)
    at.set_fontweight('bold')
    at.set_color('white')
ax2.set_title('C_INE_2 · Reincidencia\n(golden record: 140 individuos)',
              fontweight='bold', color=CRIMSON)
ax2.text(0, -1.4,
         f'El reincidente con más casos: 10 incidencias\n'
         f'(Ernesto Ruiz Flandes, Veracruz — sanción: Ninguna)',
         ha='center', fontsize=7.5, color=CRIMSON, style='italic')

# C_INE_3: Ámbito territorial (municipal/estatal/federal)
ax3 = fig.add_subplot(gs[0, 2])
ambito_col = 'Ambito Territorial' if 'Ambito Territorial' in ine.columns \
             else 'Ámbito Territorial'
if ambito_col in ine.columns:
    ambito    = safe_str(ine[ambito_col]).replace('', 'No especificado')
    ambito_ct = ambito.value_counts().head(6)
    colors_a  = [CRIMSON, TEAL, GOLD, '#5D6D7E', '#A9A9A9', '#D5D8DC']
    bars = ax3.bar(range(len(ambito_ct)), ambito_ct.values,
                   color=colors_a[:len(ambito_ct)],
                   edgecolor='white')
    ax3.set_xticks(range(len(ambito_ct)))
    ax3.set_xticklabels(ambito_ct.index, rotation=25, ha='right', fontsize=8.5)
    ax3.set_title('C_INE_3 · Ámbito territorial\nde la sanción (INE)',
                  fontweight='bold', color=SLATE)
    ax3.set_ylabel('Frecuencia')
    for bar, val in zip(bars, ambito_ct.values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', fontsize=8.5, color=SLATE,
                 fontweight='bold')

# C_INE_4: Sanción por sexo (stacked bar)
ax4 = fig.add_subplot(gs[1, 0])
sanc_sexo = pd.crosstab(
    safe_str(ine[INE_SEX]).replace('', 'NS'),
    safe_str(ine[INE_SANC]).replace('', 'No especificada')
).head(3)
sanc_sexo_pct = sanc_sexo.div(sanc_sexo.sum(axis=1), axis=0) * 100
sanc_sexo_pct = sanc_sexo_pct[sanc_sexo_pct.columns[:6]]  # top 6 sanciones
sanc_sexo_pct.plot(kind='bar', stacked=True, ax=ax4,
                   colormap='RdBu_r', edgecolor='white', width=0.6)
ax4.set_title('C_INE_4 · Distribución de sanciones\npor sexo del agresor (%)',
              fontweight='bold', color=SLATE)
ax4.set_xlabel('Sexo')
ax4.set_ylabel('% del total de sanciones')
ax4.yaxis.set_major_formatter(mtick.PercentFormatter())
ax4.tick_params(axis='x', rotation=0)
ax4.legend(title='Sanción', bbox_to_anchor=(1.01, 1),
           fontsize=7, title_fontsize=7.5)

# C_INE_5: Top reincidentes (golden record)
ax5 = fig.add_subplot(gs[1, 1:])
top_rein = golden.nlargest(10, 'n_incidencias')[
    ['nombre', 'entidad', 'n_incidencias', 'sanciones']].copy()
top_rein['nombre_corto'] = top_rein['nombre'].str.title().str[:28]
top_rein['label'] = top_rein.apply(
    lambda r: f"{r['nombre_corto']} ({r['entidad']})" if pd.notna(r['entidad'])
    else r['nombre_corto'], axis=1)
bars = ax5.barh(range(len(top_rein)), top_rein['n_incidencias'],
                color=[CRIMSON if v >= 5 else CRIMSON_L
                       for v in top_rein['n_incidencias']],
                edgecolor='white', height=0.7)
ax5.set_yticks(range(len(top_rein)))
ax5.set_yticklabels(top_rein['label'], fontsize=8.5)
ax5.set_xlabel('Número de incidencias')
ax5.set_title('C_INE_5 · Top 10 reincidentes en el golden record\n'
              '(consolidados por nombre: 162 incidencias → 140 individuos)',
              fontweight='bold', color=CRIMSON)
for bar, val in zip(bars, top_rein['n_incidencias']):
    ax5.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
             str(val), va='center', fontsize=8.5, color=CRIMSON,
             fontweight='bold')

plt.tight_layout()
guardar('fig07_consultas_ine')


Fig 7 — 5 consultas INE...
  OK  fig07_consultas_ine.png


### Fig. 8 — 5 Consultas BANAVIM (datos procesados)


In [13]:
print('Fig 8 — 5 consultas BANAVIM...')

fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(2, 3, figure=fig, wspace=0.42, hspace=0.55)
titulo_figura(fig,
    '5 Consultas sobre el BANAVIM — datos después del procesamiento',
    'Mojibake reparado con ftfy · '
    'Edades atípicas excluidas · placeholders semánticos eliminados')

# C_BV_1: Pirámide de edad por sexo
ax1 = fig.add_subplot(gs[0, :2])
bv_edad_sexo = bv[['Sexo', BV_EDAD]].copy()
bv_edad_sexo['edad_num']  = pd.to_numeric(bv_edad_sexo[BV_EDAD], errors='coerce')
bv_edad_sexo['sexo_norm'] = safe_str(bv_edad_sexo['Sexo']).str.upper()
bv_edad_sexo = bv_edad_sexo[bv_edad_sexo['edad_num'].between(14, 80)]
bv_h = bv_edad_sexo[bv_edad_sexo['sexo_norm'] == 'H']['edad_num']
bv_m = bv_edad_sexo[bv_edad_sexo['sexo_norm'] == 'M']['edad_num']
bins_edad = np.arange(14, 82, 5)
h_h, _ = np.histogram(bv_h, bins=bins_edad)
h_m, _ = np.histogram(bv_m, bins=bins_edad)
mid = (bins_edad[:-1] + bins_edad[1:]) / 2
ax1.barh(mid, -h_h, height=4, color=TEAL,   edgecolor='white', alpha=0.85, label='Hombre')
ax1.barh(mid,  h_m, height=4, color=CRIMSON, edgecolor='white', alpha=0.7,  label='Mujer')
ax1.axvline(0, color=SLATE, lw=0.8)
ax1.set_title('C_BV_1 · Pirámide de edad por sexo del agresor\n'
              '(BANAVIM · mojibake reparado · outliers excluidos)',
              fontweight='bold', color=TEAL)
ax1.set_xlabel('← Hombres  |  Mujeres →')
ax1.set_ylabel('Edad')
ax1.xaxis.set_major_formatter(
    mtick.FuncFormatter(lambda x, _: f'{abs(int(x)):,}'))
ax1.legend()
ax1.text(0.01, 0.97,
         f'Los agresores hombres\ntienen en promedio\n{bv_h.mean():.0f} años',
         transform=ax1.transAxes, va='top', fontsize=8, color=TEAL,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# C_BV_2: Tipos de arma
ax2 = fig.add_subplot(gs[0, 2])
cols_arma = [c for c in bv.columns if any(k in c for k in
    ['Chacos','Macanas','OtraArmaBlanca','ObjetoPunzo',
     'Machete','Proyectil','ArmaFuego','OtraFuego'])]
if cols_arma:
    arma_tipos  = {c.replace('_',' ').replace('Arma','')[:18]:
                   (safe_str(bv[c]).str.upper() == 'SI').sum()
                   for c in cols_arma}
    arma_df     = pd.Series(arma_tipos).sort_values()
    arma_df     = arma_df[arma_df > 0]
    colors_arma = [CRIMSON if 'Fuego' in k else '#E07070' for k in arma_df.index]
    ax2.barh(range(len(arma_df)), arma_df.values,
             color=colors_arma, edgecolor='white', height=0.7)
    ax2.set_yticks(range(len(arma_df)))
    ax2.set_yticklabels(arma_df.index, fontsize=8)
    ax2.set_title('C_BV_2 · Tipos de arma portada\n(BANAVIM)',
                  fontweight='bold', color=CRIMSON)
    ax2.set_xlabel('Frecuencia (log)')
    ax2.set_xscale('log')

# C_BV_3: Evolución mensual (estacionalidad)
ax3 = fig.add_subplot(gs[1, 0])
bv_fecha = pd.to_datetime(safe_str(bv['fecha_registro']), errors='coerce')
bv_mes   = bv_fecha.dt.month.value_counts().sort_index()
meses    = ['Ene','Feb','Mar','Abr','May','Jun',
            'Jul','Ago','Sep','Oct','Nov','Dic']
ax3.fill_between(range(1, 13), bv_mes.reindex(range(1,13)).fillna(0).values,
                 color=TEAL, alpha=0.35)
ax3.plot(range(1, 13), bv_mes.reindex(range(1,13)).fillna(0).values,
         color=TEAL, lw=2.5, marker='o', ms=6,
         markeredgecolor='white', markeredgewidth=1.5)
ax3.set_xticks(range(1, 13))
ax3.set_xticklabels(meses, fontsize=8)
ax3.set_title('C_BV_3 · Estacionalidad de casos\n(BANAVIM 2020–2022)',
              fontweight='bold', color=TEAL)
ax3.set_ylabel('Casos registrados')
ax3.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'{int(x):,}'))

# C_BV_4: Consumo de sustancias
ax4 = fig.add_subplot(gs[1, 1])
sust_cols = {
    'Alcohol':        'Droga_Alcohol',
    'Droga médica':   'Droga_DrogaPorIndicacion Medica'
                      if 'Droga_DrogaPorIndicacion Medica' in bv.columns
                      else 'Droga_DrogaPorIndicaciÃ³n MÃ©dica',
    'Drogas ilegales':'Droga_Drogas Ilegales',
}
pcts = {}
for label, col in sust_cols.items():
    if col in bv.columns:
        pcts[label] = (safe_str(bv[col]).str.upper() == 'SI').mean() * 100
if pcts:
    pcts_s   = pd.Series(pcts).sort_values()
    colors_s = [GOLD, TEAL, CRIMSON][:len(pcts_s)]
    bars = ax4.barh(range(len(pcts_s)), pcts_s.values,
                    color=colors_s, edgecolor='white', height=0.6)
    ax4.set_yticks(range(len(pcts_s)))
    ax4.set_yticklabels(pcts_s.index, fontsize=9)
    ax4.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax4.set_title('C_BV_4 · % casos con consumo\nde sustancias (BANAVIM)',
                  fontweight='bold', color=GOLD)
    for bar, val in zip(bars, pcts_s.values):
        ax4.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9, color=GOLD,
                 fontweight='bold')

# C_BV_5: ¿La víctima conoce al agresor?
ax5 = fig.add_subplot(gs[1, 2])
if 'Conoce al Agresor' in bv.columns or 'Conoce al agresor' in bv.columns:
    col_conoce = 'Conoce al Agresor' if 'Conoce al Agresor' in bv.columns \
                 else 'Conoce al agresor'
    conoce    = safe_str(bv[col_conoce]).str.upper().replace('', np.nan).dropna()
    conoce_ct = conoce.value_counts().head(4)
    colors_c  = [TEAL, CRIMSON, GOLD, SLATE][:len(conoce_ct)]
    wedges, texts, auto = ax5.pie(
        conoce_ct.values, labels=conoce_ct.index,
        autopct='%1.0f%%', colors=colors_c,
        startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=1.5),
        textprops=dict(fontsize=8))
    for at in auto:
        at.set_fontsize(8.5)
        at.set_fontweight('bold')
        at.set_color('white')
    ax5.set_title('C_BV_5 · ¿La víctima conoce\nal agresor? (BANAVIM)',
                  fontweight='bold', color=TEAL)
else:
    ax5.text(0.5, 0.5, 'Campo no disponible\nen esta versión del BANAVIM',
             ha='center', va='center', fontsize=10, color=SLATE)
    ax5.set_title('C_BV_5 · Relación víctima-agresor', fontweight='bold')

plt.tight_layout()
guardar('fig08_consultas_banavim')


Fig 8 — 5 consultas BANAVIM...
  OK  fig08_consultas_banavim.png


## 5. Figuras — Consultas multi-fuente

Las figuras 9 y 10 cruzan simultáneamente INE y BANAVIM a través del perfil integrado generado por el Sorted Neighbourhood Method en `fusion.ipynb`.


### Fig. 9 — Consultas multi-fuente (1–5)


In [14]:
print('Fig 9 — Multi-fuente 1-5...')

fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(2, 3, figure=fig, wspace=0.42, hspace=0.55)
titulo_figura(fig,
    'Consultas multi-fuente (1–5) — INE + BANAVIM integradas',
    'Las siguientes consultas cruzan simultáneamente ambas fuentes · '
    'Integración consolidada vía Sorted Neighbourhood Method')

# C_MULTI_1: Sanciones vs casos por estado (divergente)
ax1 = fig.add_subplot(gs[0, :2])
comp = (perfil.groupby('estado_ine')
        .agg(sanciones=('n_sanciones_ine','sum'),
             casos=('n_casos_banavim','sum'))
        .sort_values('casos', ascending=True)
        .reset_index())
y = np.arange(len(comp))
ax1_twin = ax1.twiny()
ax1.barh(y, comp['sanciones'], color=CRIMSON, edgecolor='white',
         height=0.4, label='Sanciones INE', alpha=0.9)
ax1_twin.barh(y + 0.4, comp['casos'], color=TEAL, edgecolor='white',
              height=0.4, label='Casos BANAVIM', alpha=0.7)
ax1.set_yticks(y + 0.2)
ax1.set_yticklabels(comp['estado_ine'], fontsize=9)
ax1.set_xlabel('Sanciones INE', color=CRIMSON, fontsize=9)
ax1_twin.set_xlabel('Casos BANAVIM (escala log)', color=TEAL, fontsize=9)
ax1_twin.set_xscale('log')
handles = [mpatches.Patch(color=CRIMSON, label='Sanciones INE'),
           mpatches.Patch(color=TEAL,    label='Casos BANAVIM')]
ax1.legend(handles=handles, loc='lower right', fontsize=8)
ax1.set_title('C_MULTI_1 · Sanciones formales vs casos registrados\n'
              'por estado (ambas fuentes simultáneamente)',
              fontweight='bold', color=SLATE)

# C_MULTI_2: Brecha institucional (casos por sanción)
ax2 = fig.add_subplot(gs[0, 2])
brecha = comp.copy()
brecha['ratio'] = brecha['casos'] / brecha['sanciones'].replace(0, np.nan)
brecha = brecha.dropna().sort_values('ratio', ascending=True)
norm_b   = mcolors.Normalize(vmin=0, vmax=brecha['ratio'].max())
colors_b = [CMAP_RISK(norm_b(v)) for v in brecha['ratio']]
bars = ax2.barh(range(len(brecha)), brecha['ratio'],
                color=colors_b, edgecolor='white', height=0.7)
ax2.set_yticks(range(len(brecha)))
ax2.set_yticklabels(brecha['estado_ine'], fontsize=8.5)
ax2.set_xlabel('Casos BANAVIM por cada sanción INE')
ax2.set_title('C_MULTI_2 · Brecha institucional:\n'
              'casos sin sanción formal por estado',
              fontweight='bold', color=SLATE)
ax2.text(0.98, 0.02,
         'Quintana Roo:\n1 255 casos por sanción',
         transform=ax2.transAxes, ha='right', va='bottom',
         fontsize=8, color=CRIMSON, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

# C_MULTI_3: Completitud de campos de fusión
ax3 = fig.add_subplot(gs[1, 0])
try:
    ine_f = pd.read_csv('../Data/fusion_outputs/ine_fusion_2020_2022.csv',
                        low_memory=False)
    bv_f  = pd.read_csv('../Data/fusion_outputs/banavim_fusion_2020_2022.csv.gz',
                         low_memory=False)
    campos  = ['sexo', 'estado', 'municipio', 'vinculo_victima']
    comp_ine = [ine_f[c].notna().mean()*100 if c in ine_f.columns else 0 for c in campos]
    comp_bv  = [bv_f[c].notna().mean()*100  if c in bv_f.columns  else 0 for c in campos]
    x = np.arange(len(campos))
    ax3.bar(x - 0.2, comp_ine, 0.35, color=CRIMSON, edgecolor='white', label='INE', alpha=0.9)
    ax3.bar(x + 0.2, comp_bv,  0.35, color=TEAL,    edgecolor='white', label='BANAVIM', alpha=0.8)
    ax3.axhline(80, color=GOLD, lw=1.5, ls='--', alpha=0.7, label='Umbral 80%')
    ax3.set_xticks(x)
    ax3.set_xticklabels(campos, rotation=20, ha='right', fontsize=8.5)
    ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax3.set_title('C_MULTI_3 · Completitud de\ncampos de fusión (ambas fuentes)',
                  fontweight='bold', color=SLATE)
    ax3.legend(fontsize=8)
    ax3.set_ylim(0, 110)
except Exception as e:
    ax3.text(0.5, 0.5, f'Datos no disponibles\n{str(e)[:40]}',
             ha='center', va='center', fontsize=9, color=SLATE)

# C_MULTI_4: Distribución de sexo comparada (barras agrupadas %)
ax4 = fig.add_subplot(gs[1, 1])
sexo_ine = (safe_str(ine[INE_SEX]).replace('', 'NS')
            .value_counts(normalize=True).mul(100).head(3))
sexo_bv  = (safe_str(bv['Sexo']).str.upper().replace('', 'NS')
            .value_counts(normalize=True).mul(100).head(3))
idx_comun = list(set(sexo_ine.index) | set(sexo_bv.index))
comp_sexo = pd.DataFrame({'INE': sexo_ine.reindex(idx_comun, fill_value=0),
                          'BANAVIM': sexo_bv.reindex(idx_comun, fill_value=0)},
                         index=idx_comun)
comp_sexo.plot(kind='bar', ax=ax4, color=[CRIMSON, TEAL],
               edgecolor='white', width=0.6)
ax4.yaxis.set_major_formatter(mtick.PercentFormatter())
ax4.set_title('C_MULTI_4 · Distribución de sexo\ndel agresor (ambas fuentes)',
              fontweight='bold', color=SLATE)
ax4.set_xlabel('')
ax4.tick_params(axis='x', rotation=0)
ax4.legend(['INE', 'BANAVIM'], fontsize=9)

# C_MULTI_5: Edad media BANAVIM vs sanciones INE (scatter)
ax5 = fig.add_subplot(gs[1, 2])
p5 = perfil.dropna(subset=['edad_media_bv', 'n_sanciones_ine'])
sc = ax5.scatter(p5['n_sanciones_ine'],
                 p5['edad_media_bv'],
                 c=p5['pct_portaba_arma'].fillna(0),
                 cmap=CMAP_RISK, s=60, alpha=0.8,
                 edgecolors=SLATE, linewidths=0.5)
for _, row in p5.iterrows():
    ax5.annotate(row['estado_ine'][:5],
                 (row['n_sanciones_ine'], row['edad_media_bv']),
                 fontsize=6.5, ha='center', va='bottom', color=SLATE)
plt.colorbar(sc, ax=ax5, label='% con arma', shrink=0.8)
ax5.set_title('C_MULTI_5 · Edad media (BANAVIM)\nvs sanciones (INE) por estado',
              fontweight='bold', color=SLATE)
ax5.set_xlabel('Sanciones INE')
ax5.set_ylabel('Edad media del agresor (BANAVIM)')

plt.tight_layout()
guardar('fig09_consultas_multi_1_5')


Fig 9 — Multi-fuente 1-5...
  OK  fig09_consultas_multi_1_5.png


### Fig. 10 — Consultas multi-fuente (6–10)


In [15]:
print('Fig 10 — Multi-fuente 6-10...')

fig = plt.figure(figsize=(20, 12))
gs  = GridSpec(2, 3, figure=fig, wspace=0.42, hspace=0.55)
titulo_figura(fig,
    'Consultas multi-fuente (6–10) — hacia la política pública de prevención',
    'Perfil integrado INE + BANAVIM · '
    'La ciencia de datos como herramienta para hacer visible lo invisible')

# C_MULTI_6: Perfil de riesgo BANAVIM en estados con sanciones INE
ax1 = fig.add_subplot(gs[0, 0])
m6 = perfil[perfil['n_sanciones_ine'] >= 1].groupby('estado_ine').agg(
    pct_arma=('pct_portaba_arma','mean'),
    pct_alco=('pct_alcohol','mean'),
    n=('n_sanciones_ine','sum')).reset_index().sort_values('n', ascending=False)
x6 = np.arange(len(m6))
ax1.bar(x6 - 0.2, m6['pct_arma']*100, 0.35,
        color=CRIMSON, edgecolor='white', alpha=0.9, label='% con arma')
ax1.bar(x6 + 0.2, m6['pct_alco']*100, 0.35,
        color=GOLD,   edgecolor='white', alpha=0.85, label='% con alcohol')
ax1.set_xticks(x6)
ax1.set_xticklabels(m6['estado_ine'], rotation=35, ha='right', fontsize=8)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_title('C_MULTI_6 · Perfil de riesgo BANAVIM\nen estados con sanciones INE',
              fontweight='bold', color=SLATE)
ax1.legend(fontsize=8)

# C_MULTI_7: Comparativa temporal INE vs BANAVIM
ax2 = fig.add_subplot(gs[0, 1])
ine_anual_pct = (ine['anio'].value_counts(normalize=True)
                 .mul(100).reindex([2020,2021,2022]).fillna(0))
bv_anual = pd.to_datetime(safe_str(bv['fecha_registro']), errors='coerce').dt.year
bv_anual_pct = (bv_anual.value_counts(normalize=True)
                .mul(100).reindex([2020,2021,2022]).fillna(0))
years = [2020, 2021, 2022]
ax2.plot(years, ine_anual_pct.values,
         color=CRIMSON, lw=2.5, marker='o', ms=8,
         markeredgecolor='white', markeredgewidth=1.5,
         label='INE (% del total)')
ax2.plot(years, bv_anual_pct.values,
         color=TEAL, lw=2.5, marker='s', ms=8,
         markeredgecolor='white', markeredgewidth=1.5,
         label='BANAVIM (% del total)')
for y, vi, vb in zip(years, ine_anual_pct.values, bv_anual_pct.values):
    ax2.annotate(f'{vi:.0f}%', (y, vi), xytext=(0, 8),
                 textcoords='offset points', ha='center', fontsize=8, color=CRIMSON)
    ax2.annotate(f'{vb:.0f}%', (y, vb), xytext=(0, -14),
                 textcoords='offset points', ha='center', fontsize=8, color=TEAL)
ax2.set_xticks(years)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_title('C_MULTI_7 · Distribución temporal\nINE vs BANAVIM (% por año)',
              fontweight='bold', color=SLATE)
ax2.set_xlabel('Año')
ax2.legend(fontsize=8)

# C_MULTI_8: Grupos de vínculo homologados entre fuentes
ax3 = fig.add_subplot(gs[0, 2])
vhom_ine = (perfil.groupby('vinculo_grupo_ine')['n_sanciones_ine']
            .sum().sort_values(ascending=True))
vhom_bv  = (perfil.groupby('vinculo_grupo_ine')['n_casos_banavim']
            .sum().reindex(vhom_ine.index).fillna(0))
y = np.arange(len(vhom_ine))
ax3.barh(y - 0.2, vhom_ine.values, 0.35,
         color=CRIMSON, edgecolor='white', alpha=0.9)
ax3_b = ax3.twiny()
ax3_b.barh(y + 0.2, vhom_bv.values, 0.35,
           color=TEAL, edgecolor='white', alpha=0.7)
ax3.set_yticks(y)
ax3.set_yticklabels(vhom_ine.index, fontsize=8.5)
ax3.set_xlabel('Sanciones INE', color=CRIMSON, fontsize=8)
ax3_b.set_xlabel('Casos BANAVIM', color=TEAL, fontsize=8)
handles = [mpatches.Patch(color=CRIMSON, label='INE'),
           mpatches.Patch(color=TEAL,    label='BANAVIM')]
ax3.legend(handles=handles, fontsize=8, loc='lower right')
ax3.set_title('C_MULTI_8 · Grupos de vínculo\nhomologados entre fuentes',
              fontweight='bold', color=SLATE)

# C_MULTI_9: Heatmap componentes del riesgo normalizados
ax4 = fig.add_subplot(gs[1, :2])
hm_data = (perfil_idx.pivot_table(
    index='estado_ine', columns='vinculo_grupo_ine',
    values=['n_sanciones_ine','n_casos_banavim',
            'pct_portaba_arma','pct_alcohol'],
    aggfunc='mean').fillna(0))
hm_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(hm_data),
    index=hm_data.index, columns=hm_data.columns)
sns.heatmap(hm_norm, cmap=CMAP_RISK,
            linewidths=0.3, ax=ax4,
            cbar_kws={'label': 'Valor normalizado (0–1)', 'shrink': 0.8},
            xticklabels=True, yticklabels=True)
ax4.set_title('C_MULTI_9 · Todos los componentes del riesgo normalizados\n'
              '(INE + BANAVIM · integración multi-fuente)',
              fontweight='bold', color=SLATE)
ax4.tick_params(axis='x', rotation=30, labelsize=6.5)
ax4.tick_params(axis='y', rotation=0,  labelsize=8)

# C_MULTI_10: Top 5 estados prioritarios para intervención
ax5 = fig.add_subplot(gs[1, 2])
top5 = ranking.head(5).copy()
top5_colors = [CMAP_RISK(ranking['indice'].max() * (1 - i*0.12))
               for i in range(5)]
bars = ax5.barh(range(5), top5['indice'],
                color=top5_colors, edgecolor='white', height=0.7)
ax5.set_yticks(range(5))
ax5.set_yticklabels(top5['estado'], fontsize=10)
ax5.set_xlabel('Índice de riesgo integrado')
ax5.set_title('C_MULTI_10 · Top 5 estados\nprioritarios para intervención',
              fontweight='bold', color=CRIMSON)
for bar, val, estado in zip(bars, top5['indice'], top5['estado']):
    ax5.text(bar.get_width() + 0.003,
             bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9,
             fontweight='bold', color=CRIMSON)
ax5.text(0.5, -0.22,
         'El índice integra sanciones formales, volumen de casos,\n'
         'portación de armas y consumo de alcohol como señales de riesgo.',
         transform=ax5.transAxes, ha='center', fontsize=7.5,
         color='#7F8C8D', style='italic')

plt.tight_layout()
guardar('fig10_consultas_multi_6_10')


Fig 10 — Multi-fuente 6-10...
  OK  fig10_consultas_multi_6_10.png


## 6. Resumen de salida


In [16]:
graficas = list(OUT.glob('*.png'))
print(f'\n{"="*60}')
print(f'  {len(graficas)} figuras generadas en {OUT}')
for g in sorted(graficas):
    print(f'  {g.name}')
print(f'{"="*60}')



  10 figuras generadas en ../Data/graficas
  fig01_perfil_sancionados_ine.png
  fig02_tipos_violencia_sanciones.png
  fig03_distribucion_territorial.png
  fig04_vinculos_victima_agresor.png
  fig05_perfil_agresor_banavim.png
  fig06_indice_riesgo_territorial.png
  fig07_consultas_ine.png
  fig08_consultas_banavim.png
  fig09_consultas_multi_1_5.png
  fig10_consultas_multi_6_10.png
